In [ ]:
import base64
import json
import requests
import os
from dotenv import load_dotenv
from typing import Dict, Any
from IPython.display import Markdown, display, Image

# Mistral Document AI Basics

Mistral Document AI offers enterprise-level document processing, combining cutting-edge OCR technology with advanced structured data extraction. This notebook showcases a few examples of basic OCR extraction for text and images.

We will be using the `mistral-document-ai-2505` model with a few documents and images to show the capabilities of the model.

> **Note**: The Document AI endpoint on Azure Foundry cannot process sources from external URLSs, instead we show you how to encode documents and images and call the API.

## 0. Setup

From Azure AI Foundry, after clicking on "Use this model" and going through standard Foundry setup you will need the provided Endpoint, Key and Deployment Name.

The Endpoint is used as the url in the REST request, the Key is used for authentication in the header, and the Deployment Name is used in the body of the request.

In [ ]:
load_dotenv()

AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT = os.getenv("AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT")
AZURE_MISTRAL_DOCUMENT_AI_KEY = os.getenv("AZURE_MISTRAL_DOCUMENT_AI_KEY")
AZURE_AI_DEPLOYMENT_NAME = os.getenv("AZURE_AI_DEPLOYMENT_NAME")


REQUEST_HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {AZURE_MISTRAL_DOCUMENT_AI_KEY}",
}

In [ ]:
AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT

## 1. Helper Functions

In [ ]:
def encode_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode("utf-8")
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None

def testBBoxAnnotationOutput(response: Dict[str, Any]):
    for page in response.json()["pages"]:
        for image in page["images"]:
            print("page " + str(page["index"]))
            iaj = json.loads(image["image_annotation"])
            print("Image type: " , iaj["properties"]["image_type"])
            print("Short description: ", iaj["properties"]["short_description"])

def bboxannotation(encoding: str, format: str) -> Dict[str, Any]:
    bboxannotationPayload = {
    "model": f"{AZURE_AI_DEPLOYMENT_NAME}",
    "document": {
        "type": "document_url",
        "document_url": f"data:application/pdf;base64,{encoding}",
    },
    "extract_footer": True,
    "extract_header": True, 
    "include_blocks": True,
    "include_image_base64": True,
    "confidence_scores_granularity": "page",
    f"{format}": {
        "type": "json_schema",
        "json_schema": {
            "name": "string",
            "description": "string",
            "schema": {
                "properties": {
                    "image_type": {
                        "description": "The type of the image.",
                        "title": "Image Type",
                        "type": "string",
                    },
                    "short_description": {
                        "description": "A description in english describing the image.",
                        "title": "Short Description",
                        "type": "string",
                    },
                }
            },
        },
    },
}

    return bboxannotationPayload


def simple_combined_markdown(responsePage: dict) -> str:
    markdowns: list[str] = []
    image_data = {}
    for img in responsePage["images"]:
        image_data[img["id"]] = img["image_base64"]
    markdowns.append(replace_images_in_markdown(responsePage["markdown"], image_data))

    return "\n\n".join(markdowns)

def replace_images_in_markdown(markdown_str: str, images_dict: dict) -> str:
    for img_name, base64_str in images_dict.items():
        markdown_str = markdown_str.replace(
            f"![{img_name}]({img_name})", f"![{img_name}]({base64_str})"
        )
    return markdown_str

## 2. Image/Document Annotations

In addition to basic OCR functionality, Mistral Document AI has annotations that allow you to extract information from documents and images in structured json with a single call to the API. It offers two types of annotations that can be used independently, or together:

bbox_annotation: gives you the annotation of the bounding boxes extracted by the OCR model (charts/figures etc.) based on a structure that you define in the request. This is provided for every image, in every page of the document.
document_annotation: like the bbox_annotation, but for the extracted text across the entire document.
Note: Document annotations are currently limited at 8 pages. Please see the model card for the most up-to-date information on limits

In [ ]:
!wget  -P samples  https://raw.githubusercontent.com/mistralai/cookbook/refs/heads/main/mistral/ocr/receipt.png 
!wget  -P samples https://raw.githubusercontent.com/mistralai/cookbook/refs/heads/main/mistral/ocr/mistral7b.pdf

Construct the request and parse the response

In [ ]:
display(Image(filename="samples/receipt.png"))

In [ ]:
bb1Response_image = requests.post(
    url=AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT,
    json=bboxannotation(encode_file("samples/receipt.png"), 'document_annotation_format'),
    headers=REQUEST_HEADERS,
)

In [ ]:
response = bb1Response_image.json()
doc_annotation = json.loads(response["document_annotation"])
doc_annotation["properties"]["short_description"]

In [ ]:
bb1Response_image.json()

Retrieve Confidence Score

In [ ]:
print(response["pages"][0]["confidence_scores"])

Retreive Blocks based on 'include_blocks' set to True

In [ ]:
print(response["pages"][0]["blocks"][0])

In [ ]:
bb1Response_document = requests.post(
    url=AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT,
    json=bboxannotation(encode_file("samples/mistral7b.pdf"), 'bbox_annotation_format'),
    headers=REQUEST_HEADERS,
)

In [ ]:
image = bb1Response_document.json()["pages"][0]["images"][0]
iaj = json.loads(image["image_annotation"])
print(iaj["properties"]["short_description"])

In [ ]:
testBBoxAnnotationOutput(bb1Response_document)

You will notice that for every page, the API returns text data in html format, along with information about detected images.

# 3. Header/Footer Extraction Example

In [ ]:
display(Markdown(bb1Response_document.json()["pages"][6]["header"]))

In [ ]:
display(bb1Response_document.json()["pages"][4]["footer"])


# 4. Tabular Example

Next we look at a document with tabular data, for this example we are using Microsoft's 8-K filing located here: https://www.microsoft.com/en-us/investor/sec-filings 

In [ ]:
ms8kDocument = encode_file("samples/0000950170-25-100226.pdf")

In [ ]:
msRequestPayload = {
    "model": f"{AZURE_AI_DEPLOYMENT_NAME}",
    "document": {
        "type": "document_url",
        "document_url": f"data:application/pdf;base64,{ms8kDocument}",
    },
    "table_format": "html",
    "extract_header": True, 
    "extract_footer": True,
}

In [ ]:
ms8kResponse = requests.post(
    url=AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT,
    json=msRequestPayload,
    headers=REQUEST_HEADERS,
)

Selecting a page with tables

In [ ]:
display(ms8kResponse.json()["pages"][0]["header"])

In [ ]:
print(ms8kResponse.json()["pages"][0]["tables"][0]['content'])

In [ ]:
print(ms8kResponse.json()["pages"][0]["markdown"])

In [ ]:
display(Markdown(simple_combined_markdown(ms8kResponse.json()["pages"][0])))

As observed, the extracted text in the table is accurate and true to the original tabular representation.

# 5. Microsoft Word Documents

In [ ]:
wordDocument = encode_file("samples/TranscriptFY25q4.docx")

In [ ]:
wordPayload = {
    "model": f"{AZURE_AI_DEPLOYMENT_NAME}",
    "document": {
        "type": "document_url",
        "document_url": f"data:application/vnd.openxmlformats-officedocument.wordprocessingml.document;base64,{wordDocument}",
    },
    "table_format": "html",
    "extract_header": True, 
    "extract_footer": True, 
}

In [ ]:
wordResponse = requests.post(
    url=AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT,
    json=wordPayload,
    headers=REQUEST_HEADERS,
)

In [ ]:
print(wordResponse.json()["pages"][0]['markdown'])

In [ ]:
# limiting to 1000 characters for easier reading
display(Markdown(simple_combined_markdown(wordResponse.json()["pages"][0])[:1000]))

# 6. Microsoft Powerpoint Presentations

In [ ]:
!wget -P samples https://github.com/mistralai/cookbook/raw/52ed72962a4d2be2c922919d7c25016b71b900df/data/sample.pptx

In [ ]:
powerpointDocument = encode_file("samples/sample.pptx")

In [ ]:
pptxPayload = {
    "model": f"{AZURE_AI_DEPLOYMENT_NAME}",
    "document": {
        "type": "document_url",
        "document_url": f"data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{powerpointDocument}",
    },
    "include_image_base64": "false",
    "image_limit": 0,
    "table_format": "html",
    "extract_header": True, 
    "extract_footer": True, 
}

In [ ]:
pptxResponse = requests.post(
    url=AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT,
    json=pptxPayload,
    headers=REQUEST_HEADERS,
)

In [ ]:
pptxResponse.json()["pages"][0]['markdown']

In [ ]:
print(json.dumps(pptxResponse.json(), indent=4))

EPub File Format

In [ ]:
epubDocument = encode_file("samples/minimal.epub")

In [ ]:
epubPayload = {
    "model": f"{AZURE_AI_DEPLOYMENT_NAME}",
    "document": {
        "type": "document_url",
        "document_url": f"data:application/epub+zip;base64,{epubDocument}",
    },
    "include_image_base64": "false",
    "image_limit": 0,
    "table_format": "html",
    "extract_header": True, 
    "extract_footer": True, 
}

In [ ]:
epubResponse = requests.post(
    url=AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT,
    json=epubPayload,
    headers=REQUEST_HEADERS,
)

In [ ]:
epubResponse.json()["pages"][0]['markdown']

# 7. Wrap-up

Documents and images are a wealth of information, but that information is only useful if it can be extracted accurately. Mistral Document AI on Azure is a powerful tool that can help you extract text and images from documents accurately, with ease. We hope you found this notebook helpful and we look forward to seeing what you build with it!